# Content based Movie Recommender System

###  Anjor  Patil          226532
###  Aditi Lokhande    226501

### 1. DATA ==> 2. PREPROCESSING  ==> 3. MODEL

### Types of recommender system:  1. Content based,    2. Collaborative Filtering, 3. Hybrid  

In [ ]:
import pandas as pd
import numpy as np

### movies dataset 

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
movies

### credits dataset 

In [ ]:
credits = pd.read_csv("tmdb_5000_credits.csv")
credits

In [ ]:
movies.shape

In [ ]:
credits.shape

### Merge both datasets 

In [ ]:
movies = movies.merge(credits, on = 'title')
movies

In [ ]:
movies.shape

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.head(1)

### Important columns :
### genres
### id
### keywords
### title
### overview
### cast
### crew

### Create new dataframe based on important columns 

In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.head()

### Check null values 

In [ ]:
movies.isnull().sum()

Drop all null values

In [ ]:
movies.dropna(inplace = True)

In [ ]:
movies.isnull().sum()

### Check duplicate values 

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres

### We need to extract only the genres hence import ast library and use literal_eval method , create function convert

In [ ]:
import ast

In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L    

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head()

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)
movies.head()

In [ ]:
movies['cast'][0]

In [ ]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L       

### Take only first three actors 

In [ ]:

movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
movies.head()

In [ ]:
movies.crew[0]

### Extract value where job as director 

In [ ]:

def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L        

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head()

### Convert overview str to list 

In [ ]:

movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies.head()

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])
movies.head()

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies.head()

In [ ]:
new_df = movies[['movie_id', 'title', 'tags']]
new_df

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [ ]:
new_df.head()

### Import nltk portstemmer to perform stemming: used to get root value, i.e. flying ==> fly 

In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)    

In [ ]:
new_df['tags'].apply(stem)

In [ ]:
new_df['tags'][0]

In [ ]:
#convert into lowercase
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())
new_df.head()

### Text vectorization calculate similarity score between tag of one movie and another-  find common words or convert tags -> vectors...hence each movie will be a vector..i.e there are 5000 vectors this technique is called Bag Of Words 

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000, stop_words = 'english')

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names()

In [ ]:
ps.stem('loving')

In [ ]:
stem('In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron')

### In high dimentions euclidean distance is not a good measure...hence use cosine distance ...dist is inversley proportional to simarity 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
# sorted(list(enumerate(similarity[1])), reverse = True, key = lambda x:x[1])[1:6] #convert list of tuples and sort without changing the index similarity 

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    #convert list of tuples and sort without changing the index similarity
    movies_list = sorted(list(enumerate(distances)), reverse = True, key = lambda x:x[1])[1:6] 
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
        
    

In [ ]:
recommend('John Carter')

In [ ]:
recommend('Avatar')

In [ ]:
new_df.iloc[1216].title